# YOLO Mask Overlay Notebook

Lightweight pipeline:
1. Upload a YOLO model (`.pt`) - prints the available classes.
2. Upload an MP4.
3. Pick which classes to keep.
4. Choose a temporal smoothing window.
5. Render an output MP4 with segmentation masks overlaid only for the selected classes. Masks are temporally smoothed across the chosen window.

In [ ]:
# Install deps (skip if already present)
%pip install -q ultralytics opencv-python ipywidgets numpy

In [ ]:
import os
import tempfile
from collections import defaultdict, deque
from pathlib import Path

import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Video, FileLink, clear_output
from ultralytics import YOLO

WORK_DIR = Path(tempfile.mkdtemp(prefix="yolo_overlay_"))
STATE = {"model": None, "model_path": None, "video_path": None, "names": {}}
print(f"Working directory: {WORK_DIR}")

## 1. Upload YOLO model and inspect classes

In [ ]:
model_uploader = widgets.FileUpload(accept=".pt", multiple=False, description="Upload .pt")
model_status = widgets.Output()

def _on_model_upload(change):
    with model_status:
        clear_output()
        if not model_uploader.value:
            return
        item = list(model_uploader.value)[0]
        if isinstance(item, dict):
            name, content = item["name"], item["content"]
        else:
            name, content = item, model_uploader.value[item]["content"]
        path = WORK_DIR / name
        path.write_bytes(content)
        STATE["model_path"] = str(path)
        STATE["model"] = YOLO(str(path))
        STATE["names"] = STATE["model"].names
        print(f"Loaded {name}")
        print("\nAvailable classes:")
        for idx, cname in STATE["names"].items():
            print(f"  {idx:>3}: {cname}")
        task = getattr(STATE["model"], "task", None)
        if task and task != "segment":
            print(f"\nWarning: model task is '{task}'. Pixel masks require a segmentation model (e.g. yolov8n-seg.pt).")
            print("Detection-only models will fall back to filled bounding boxes.")

model_uploader.observe(_on_model_upload, names="value")
display(model_uploader, model_status)

## 2. Upload video

In [ ]:
video_uploader = widgets.FileUpload(accept=".mp4,.mov,.avi,.mkv", multiple=False, description="Upload video")
video_status = widgets.Output()

def _on_video_upload(change):
    with video_status:
        clear_output()
        if not video_uploader.value:
            return
        item = list(video_uploader.value)[0]
        if isinstance(item, dict):
            name, content = item["name"], item["content"]
        else:
            name, content = item, video_uploader.value[item]["content"]
        path = WORK_DIR / name
        path.write_bytes(content)
        STATE["video_path"] = str(path)
        cap = cv2.VideoCapture(str(path))
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        print(f"Loaded {name}: {w}x{h}, {fps:.2f} fps, {n} frames")

video_uploader.observe(_on_video_upload, names="value")
display(video_uploader, video_status)

## 3. Pick classes and smoothing window

`Smoothing window` = number of frames averaged per tracked instance. `1` disables smoothing. The mask itself is averaged across the window, so the overlay reflects the smoothed shape.

In [ ]:
if not STATE["names"]:
    raise RuntimeError("Upload a model first.")

class_select = widgets.SelectMultiple(
    options=[(f"{i}: {n}", i) for i, n in STATE["names"].items()],
    rows=min(12, len(STATE["names"])),
    description="Classes:",
)
smoothing_slider = widgets.IntSlider(value=5, min=1, max=31, step=1, description="Smooth (frames):")
conf_slider = widgets.FloatSlider(value=0.25, min=0.05, max=0.9, step=0.05, description="Conf:")
alpha_slider = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description="Mask alpha:")
display(class_select, smoothing_slider, conf_slider, alpha_slider)

## 4. Run inference and write output

In [ ]:
def _color_for(class_id):
    rng = np.random.default_rng(int(class_id) * 9973 + 7)
    return tuple(int(c) for c in rng.integers(64, 255, size=3))

def run(model, video_path, selected_ids, window, conf, alpha, out_path):
    selected_ids = set(int(i) for i in selected_ids)
    if not selected_ids:
        raise ValueError("Select at least one class.")

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))

    history = defaultdict(lambda: deque(maxlen=max(1, int(window))))
    is_seg = getattr(model, "task", None) == "segment"

    progress = widgets.IntProgress(value=0, min=0, max=max(total, 1), description="Rendering:")
    label = widgets.Label(value="")
    display(widgets.HBox([progress, label]))

    frame_idx = 0
    for result in model.track(source=video_path, stream=True, persist=True, conf=conf, classes=list(selected_ids), verbose=False):
        frame = result.orig_img.copy()
        boxes = result.boxes
        masks = result.masks if is_seg else None
        overlay = np.zeros_like(frame, dtype=np.float32)
        weight = np.zeros((h, w), dtype=np.float32)

        if boxes is not None and len(boxes) > 0:
            cls = boxes.cls.cpu().numpy().astype(int)
            ids = boxes.id.cpu().numpy().astype(int) if boxes.id is not None else np.arange(len(cls))
            xyxy = boxes.xyxy.cpu().numpy().astype(int)
            mask_data = masks.data.cpu().numpy() if masks is not None else None

            for i in range(len(cls)):
                c = int(cls[i])
                if c not in selected_ids:
                    continue
                track_id = int(ids[i])
                key = (track_id, c)

                if mask_data is not None:
                    m = mask_data[i]
                    if m.shape != (h, w):
                        m = cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR)
                    m = m.astype(np.float32)
                else:
                    x1, y1, x2, y2 = xyxy[i]
                    m = np.zeros((h, w), dtype=np.float32)
                    m[max(0, y1):max(0, y2), max(0, x1):max(0, x2)] = 1.0

                history[key].append(m)
                smoothed = np.mean(np.stack(history[key], axis=0), axis=0)
                binary = (smoothed >= 0.5).astype(np.float32)

                color = np.array(_color_for(c), dtype=np.float32)
                overlay += binary[..., None] * color
                weight += binary

        if weight.max() > 0:
            w_safe = np.clip(weight, 1.0, None)[..., None]
            color_layer = overlay / w_safe
            mask_any = (weight > 0)[..., None].astype(np.float32)
            blended = frame.astype(np.float32) * (1 - alpha * mask_any) + color_layer * (alpha * mask_any)
            frame = np.clip(blended, 0, 255).astype(np.uint8)

        writer.write(frame)
        frame_idx += 1
        if frame_idx % 5 == 0 or frame_idx == total:
            progress.value = min(frame_idx, progress.max)
            label.value = f"frame {frame_idx}/{total}"

    writer.release()
    progress.value = progress.max
    label.value = f"done ({frame_idx} frames)"
    return out_path

if STATE["model"] is None or STATE["video_path"] is None:
    raise RuntimeError("Upload both model and video first.")

out_path = WORK_DIR / "output_overlay.mp4"
run(
    model=STATE["model"],
    video_path=STATE["video_path"],
    selected_ids=class_select.value,
    window=smoothing_slider.value,
    conf=conf_slider.value,
    alpha=alpha_slider.value,
    out_path=out_path,
)
print(f"Wrote {out_path}")

## 5. Preview / download

In [ ]:
out_path = WORK_DIR / "output_overlay.mp4"
display(FileLink(str(out_path)))
Video(str(out_path), embed=False)